# Daisyworld From Scratch

<div align="center">
    <img src="../docs/assets/daisyworld_defaults.png">
</div>


## Summary

Daisyworld is a simple, 0-dimensional model of planetary climate regulation via uncoordinated control. Any particular instance is defined by three state values that change over time: the ground covered by two different species of daisies, $\alpha_d$ and $\alpha_l$, and stellar luminosity $L$, a unitless forcing factor that determines how much incident stellar radiation reaches the planet.

To run the model, we need three differential equations to describe how the three state values change over time. But one of these, describing how $L$ changes, is just a constant value indicating a simple linear ramp as $L$ increases (typically until reaching some maximum $L_{max}$). The other two are actually the same equation that take different values, one for each of the two daisy species in a default Daisyworld model. Five other equations define how we get the inputs to the three differential equations from Daisyworld parameters.

In this tutorial we will implement each of these equations in turn to build a Daisyworld from scratch. 

## Equations reference

_These are here to refer to while implementing Daisyworld. For more details, check out [the paper](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1600-0889.1983.tb00031.x) or [Lovelock's archived version](https://www.jameslovelock.org/biological-homeostasis-of-the-global-environment-the-parable-of-daisyworld/)_.

### Rate of change in stellar luminosity, the forcing factor

If $L < L_{max}$:

$$
\frac{dL}{dt} = \Delta L
$$
<div align="right">
    (0a)
</div>


Otherwise:

$$
\frac{dL}{dt} = 0
$$
<div align="right">
    (0b)
</div>


Where $L$ is the stellar luminosity and $\Delta L$ is the amount of change per unit time. To get the value of $L$ at the next time step $t_{next}$, increment the current value $L_{t_{now}}$ by the rate of change proportional to the time step size $\Delta t$, that is $L_{t_{next}} = L_{t_{now}} + \Delta t \frac{dL}{dt}$. In this tutorial we'll use a value of $\delta T$ for both equation 0 and 1, but note that in the implementation `dw.simple_dw` in this repository there is no explicit $\Delta t$ used for changes in L, just a simple linear ramp based on the starting and end values of L. 

### Rate of change in daisy population (ground cover)

$$
\frac{d\alpha_k}{dt} = \alpha_k (x\beta_k - \gamma)
$$
<div align="right">
    (1)
</div>

Where $\alpha_k$ is the ground coverage of daisy species $k$, $\beta_k$ is the temperature dependent growth rate for daisy speices $k$ (equation 3), and $\gamma$ is a constant proportional death rate shared by both daisy species, _e.g._ 0.05 for 5% loss per unit time. $x$ is the free ground available for daisies to grow on that is both capable of supporting daisies and not occupied by either daisy species(equation 2).

### Ground available for daisy growth

$$
x = p - \alpha_{d} - \alpha_{l}
$$
<div align="right">
    (2a)
</div>

Where $p$ is the proportion of land that is arable, or capable of supporting daisies. $\alpha_d$ and $\alpha_l$ are the proportions of ground covered by dark and light species, respectively. The original Daisyworld had only two daisy species, black and white (actually both different shades of gray), but in principle we could have many species of daisy each with a different albedo. The general equation for calculating $x$ is to subtract the sum of ground coverage for all $K$ daisy species from the proportion of ground that is arable, $p$.

$$
x = p - \sum_k^K{\alpha_k}
$$
<div align="right">
    (2b)
</div>

### Temperature-dependent growth rate

When $g(T_o-T_k)^2 \leq 1.0$:

$$
\beta_k = 1 - g(T_{o} - T_k)^2
$$
<div align="right">
    (3a)
</div>

and otherwise 
$$
\beta_k = 0
$$
<div align="right">
    (3b)
</div>

Where $\beta_k$ is the temperature-dependent growth rate for daisy species $k$, $T_o$ is the optimal temperature in Kelvin (295.5 K), $T-k$ is the local temperature experienced by daisy species $k$, and $g$ is a constant that determines the range of temperatures that allow daisies to grow. We'll use a default value of $g$ that allows daisy growth from 278 K to 313 K is $g = 1/17.5^2 \approx 0.03625$.

### Effective temperature 

$$
T_e = \left (\frac{SL(1-A)}{\sigma} \right )^{1/4}
$$
<div align="right">
    (4)
</div>

Where $T_e$ is the effective temperature of Daisyworld, $S$ is the strength of stellar radiation in Watts per square meter (we use a value of 1000 $\frac{W}{m^2}$), $L$ is unitless luminosity which we use to increase stellar radiation over time, $\sigma$ is the [Stefan-Boltzmann constant](https://en.wikipedia.org/wiki/Stefan%E2%80%93Boltzmann_law) with units of Watts divided by the product of meters squared and degrees Kelvin raised to the power of 4 , $\approx 5.67\times10^-8 \frac{W}{m^2K^4}$

The effective temperature $T_e$ ensures the energy balance of Daisyworld, as the amount of energy due to incident stellar radiation must be re-radiated by Daisyworld.

### Planetary albedo

$$
A = \alpha_g A_g + \alpha_d A_d + \alpha_l A_l
$$
<div align="right">
    (5a)
</div>

The planetary albedo $A$ determines how much stellar radiation is reflected back into space by Daisyworld. $A_g$ and $\alpha_g$ are the bare ground albedo and proportion of ground which is bare, respectively. $\alpha_d$ is the proportion of ground covered by dark and $\alpha_l$ the proportion covered by light daisies, while $A_d$ and $A_l$ are the respective albedos of each daisy species. The general equation, which covers situations with greater than or fewer than two daisy species is the sum of all daisy albedos weighted by the ground covered by each. Note that when $p$ is 1, $x$ is equal to $a_g$, but if $p$ is between 0 and 1 then $a_g = (1-p) + x$.

$$
A = \alpha_g A_g + \sum_k^K{\alpha_k A_k}
$$
<div align="right">
    (5b)
</div>

### Local temperature (experienced by each daisy species)

$$
T_k = \left( q(A - A_k) + T_e^4 \right )^{1/4}
$$
<div align="right">
    (6)
</div>

The local temperature experience by each daisy species is $T_k$, and calculated using equation 6 above. The only new variable to consider here is $q$, which has units of $\frac{1}{K^4}$ and determines the temperature difference between regions of Daisyworld with different albedos. As noted in Watson and Lovelock's 1983 paper, for $q=0$, there is perfect conduction and all of Daisyworld experiences the same temperature (and consequently, the same growth rate for all daisy species). A value greater than $\frac{SL}{\sigma}$ would mean that heat flows from the higher albedo (cooler) regions to the darker (hotter) regions: not a realistic scenario. We'll use $q = 0.2 \frac{S}{\sigma}$, similar to the value used in the 1983 paper. 


In [ ]:
"""
We'll use lambda syntax here to make the mapping from equations to code simple and clear
lambda syntax is a quick way to define short functions in Python, without using the `def function(inputs):` 
and indentation blocks that we normally use for functions.

The syntax is
```
function_name = lambda input_a, input_b: input_a + input_b
```

The part that comes after the colon defines some single-line operation on the inputs. 
You don't have to use a `return` statement like in a normal Python function.
"""

# Comments can use the `#` sign (for single-line comments) or three double quotation marks as above (for blocks)
## Step A: calculate available ground and total bare ground

# to calculate the bare ground area we just subtract the sum of daisy coverage from 1.0

# following equation 2a
calculate_x = lambda alpha_d, alpha_l: p - alpha_d - alpha_l

# seperate function for total bare ground, for cases where p is less than 1.0
calculate_bare = lambda alpha_d, alpha_l: 1 - alpha_d - alpha_l

## Step B: calculate total albedo

# now that we have values for daisy and bare ground area, we can 
# following equation 5a
calculate_albedo = lambda alpha_g, alpha_d, alpha_l: A_g*alpha_g + A_d*alpha_d + A_l*alpha_l

## Step C: calculate effective temperature
# following equation 4
calculate_temp_e = lambda L, A: ((S*L*(1-A)) / sigma)**(1/4.)

## Step D: calculate local temperature

# following equation 6
calculate_temp_k = lambda A, A_k, T_e: (q*(A - A_k) + (T_e)**4)**(1/4.)

## Step E: calculate temperature-dependent growth rate 

# from equation 3
# there is no negative 'growth' rate. Instead, mortality is captured in the $\gamma$ parameter
calculate_beta_k = lambda T_k: max([0, 1.0 - g*(T_o-T_k)**2])

## Step F: calculate the change in daisy species 

# following equation 1
calculate_dalpha_dt = lambda alpha_k, x, beta_k: alpha_k*(x*beta_k - gamma)

## Step G: calculate the change in luminosity

# it's just a constant!
# following equation 0
calculate_dL_dt = lambda: dL

## Step H: Putting it all together for one step of Daisyworld

"""
The full update and epoch loop is a little more complicated, so we'll use normal Python function syntax.

It's a little more verbose than the lambda syntax used above.

```
def function_name(input_a, input_b):
    return input_a + input_b
```

Functions are denoted by `def`, then the function name, followed by function inputs in parenthenses. 
You have to include a colon after `def function_name(inputs)`, and the next line should be indented.

Indentation is used to define scope blocks in Python. The function code all starts at the same indentation level,
and to signify the end of code that belongs to the function, go back to the original indentation level.

Unlike lambda syntax, you have to specify what you want to return with an explicit `return` statement.


You can also use type hints as in the example and functions below. These types aren't enforced by Python,
but it can be useful. 

```
# including type hints
def function_name(input_a: float, input_b: float)-> float:
    return input_a + input_b
```

The first function, `make_dw_update_function`, returns another function instead of an output variable. 
This allows you to use functions to make functions, and is a basic pattern in 'functional programming' 


It's also useful to add docstrings to your functions, which are returned when someone calls `help` on the function
(_e.g._ `help(function_name)`). It's good practice to write informative docstrings so that your code is more readable
and easier to use and understand (including by you, when you go back to code you wrote a long time ago).

I've left the docstrings below as an exercise for the reader. What do you think should go in a good docstring?
"""


def make_dw_update_function(\
        calc_x: callable, calc_bare: callable,
        calc_albedo: callable, calc_temp_e: callable,
        calc_temp_k: callable, calc_beta_k: callable,
        calc_dalpha_dt: callable, calc_dL_dt: callable,
        dt: float) -> callable:
    """
    Write a description of the function here, to be returned when users call `help(function_name)`
    """
    def dw_update_function(L: float, alpha_d: float, alpha_l: float, total_t: float) -> tuple[float]:
        """
        Write a description of the function here, to be returned when users call `help(function_name)`
        """
        alpha_g = calc_bare(alpha_d, alpha_l)
        x = calc_x(alpha_d, alpha_l)

        A = calc_albedo(alpha_g, alpha_d, alpha_l)
        T_e = calc_temp_e(L, A)
        T_d = calc_temp_k(A, A_d, T_e)
        T_l = calc_temp_k(A, A_l, T_e)

        beta_d = calc_beta_k(T_d)
        beta_l = calc_beta_k(T_l)

        da_d_dt = calc_dalpha_dt(alpha_d, x, beta_d)
        da_l_dt = calc_dalpha_dt(alpha_l, x, beta_l)
        dL_dt = calc_dL_dt()

        alpha_d_0 = 1.0 * alpha_d
        alpha_l_0 = 1.0 * alpha_l
        
        alpha_d = alpha_d + dt * da_d_dt
        alpha_l = alpha_l + dt * da_l_dt

        L = min((L + dt * dL_dt, L_max))
        
        total_t = total_t + dt

        return L, alpha_d, alpha_l, total_t, T_e

    return dw_update_function

def run_epoch(\
        dw_update_function: callable, L: float, \
        alpha_d: float, alpha_l: float, \
        epoch_duration: int=6000) -> tuple[list]:
    """
    Write a description of the function here, to be returned when users call `help(function_name)`
    """
    
    alpha_ls = []
    alpha_ds = []
    Ls = []
    time_t = []

    # trajectory of dw effective temperature
    temps = []
    
    total_time = 0.
    
    for epoch_count in range(epoch_duration):

        L, alpha_d, alpha_l, total_time, temp_e = dw_update_function(L, alpha_d, alpha_l, total_time)


        # multiplying by 1.0 is a lazy way to copy a value
        Ls.append(1. * L)
        alpha_ls.append(1. * alpha_l)
        alpha_ds.append(1. * alpha_d)
        time_t.append(1. * total_time)
        temps.append(1. * temp_e)
        
    return Ls, alpha_ds, alpha_ls, time_t, temps

## Setting up: Daisyworld constant parameters

The parameters defined in the cell below determine how Daisyworld works, but do not change over the course of a Daisyworld trajectory.

You can modify these constants to change the behaviour of a run. 

In [ ]:
# arable land: proportion of Daisyworld that can support daisies
# default value 1.0
p = 1.0

# initial values for proportional coverage of available land by daisies
# for two daisy species, the sum must be less than 1.0
a_l_initial = 0.2
a_d_initial = 0.2

# albedos for bare ground, light daisies, and dark daisies
# default values are 0.5, 0.75, and 0.25, resepctively
A_g = 0.5
A_l = 0.75
A_d = 0.25

# optimal temperature T_optim
T_o = 295.5

# temperature range constant (from equation 3a)
temp_range_margin = 17.5
g = 1/temp_range_margin**2
# or, approximately
# g = 0.03625

# proportional daisy mortality rate
gamma = 0.05

# baseline stellar output 
# default value is 1000 (W/m^2)
S = 1000

# Stefan-Boltzmann constant
sigma = 5.67e-8

# heat transfer coefficient
q = 0.2 * S / sigma

# initial, minimum, and maximum values for luminosity L
# defaults are 0.7, 0.7, and 2.0
L_initial = 0.6
L_min = 0.6
L_max = 2.0

# time step size $\delta t$
dt = 0.1

# change in luminosity $\delta L$
# epoch length is not in any particular unit of time
epoch_length = 6000
dL = (L_max - L_initial) / (dt * epoch_length)


## Daisyworld variables initial conditions and update setup


### Initial conditions
These three variables, $L$, $\alpha_d$, and $\alpha_l$, change over the course of a run and define the state of a Daisyworld instance at any given point in time. 

### update setup

The update step combines all the intermediate steps A through G into one function

In [ ]:
# initial conditions

alpha_d = a_d_initial * p
alpha_l = a_l_initial * p 
L = L_initial

# maked the update function
dw_update_function = make_dw_update_function(\
        calc_x=calculate_x, calc_bare=calculate_bare,
        calc_albedo=calculate_albedo, calc_temp_e=calculate_temp_e,
        calc_temp_k=calculate_temp_k, calc_beta_k=calculate_beta_k,
        calc_dalpha_dt=calculate_dalpha_dt, calc_dL_dt=calculate_dL_dt,
        dt=dt)

In [ ]:
# run Daisyworld

Ls, alpha_ds, alpha_ls, time_t, temps = run_epoch(\
        dw_update_function, L, \
        alpha_d, alpha_l, \
        epoch_length*2)
    

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,1, figsize=(8,4))
ax2 = ax.twinx()
ax.plot(time_t, Ls, "-.", alpha=0.5, label="luminosity")
ax2.plot(time_t, temps, "--",color="k", alpha=0.5, label=r"$T_e$, effective temperature")
ax.plot(time_t, alpha_ds, label=r"$\alpha_d$, dark daisies")
ax.plot(time_t, alpha_ls, label=r"$\alpha_l$, light daisies")
plt.title("Daisyworld run")

ax2.axis((min(time_t)-10, max(time_t)+10, 243, 380))
ax.axis((min(time_t)-10, max(time_t)+10, -0.1, 2.10))
fig.legend(loc="upper left")
plt.show()